In [2]:
# ============================================================
# CELL 1: Imports + robust geometry loading
# ============================================================

from __future__ import annotations

import os
import math
import time
import json
import random
import warnings
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np

from shapely.geometry import (
    Polygon,
    MultiPolygon,
    LineString,
    MultiLineString,
)
from shapely import wkt as shapely_wkt

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Helpers for geometry conversion
# ------------------------------------------------------------

def _linestring_to_polygon(geom):
    """
    Convert a LineString that represents a ring into a Polygon.
    Safely handles empty / degenerate line strings.
    """
    if geom is None or geom.is_empty:
        raise ValueError("empty LineString")

    coords = list(geom.coords)

    if len(coords) == 0:
        raise ValueError("LineString has no coordinates")

    # Need at least 3 points before closure
    if len(coords) < 3:
        raise ValueError(f"LineString too short for polygon conversion: {len(coords)} coords")

    if coords[0] != coords[-1]:
        coords.append(coords[0])   # force close the ring

    # Polygon ring needs at least 4 coordinates after closing
    if len(coords) < 4:
        raise ValueError(f"closed ring too short: {len(coords)} coords")

    poly = Polygon(coords)

    if poly.is_empty:
        raise ValueError("converted polygon is empty")

    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        raise ValueError("polygon became empty after validity fix")

    return poly


def _geom_to_polygon(geom):
    """
    Convert supported geometry types into Polygon / MultiPolygon.
    """
    if geom is None or geom.is_empty:
        raise ValueError("empty geometry")

    if geom.geom_type in ("Polygon", "MultiPolygon"):
        return geom

    if geom.geom_type == "LineString":
        return _linestring_to_polygon(geom)

    if geom.geom_type == "MultiLineString":
        polys = []
        for ls in geom.geoms:
            try:
                poly = _linestring_to_polygon(ls)
                polys.append(poly)
            except Exception:
                continue

        if not polys:
            raise ValueError("MultiLineString has no valid polygonal parts")

        if len(polys) == 1:
            return polys[0]
        return MultiPolygon(polys)

    raise ValueError(f"Cannot convert {geom.geom_type} to Polygon")


def _parse_tsv_line(line: str):
    """
    Parse one line of format:

        <id> <WKT> [key#value,key#value,...]

    Returns:
        feat_id, wkt_str, tag_dict
    """
    line = line.rstrip("\n").strip()
    if not line:
        raise ValueError("empty line")

    # First whitespace splits ID from the rest
    first_space = -1
    for ci, ch in enumerate(line):
        if ch in (" ", "\t"):
            first_space = ci
            break

    if first_space == -1:
        raise ValueError("no whitespace found — cannot split ID from WKT")

    feat_id = line[:first_space].strip()

    # Tags start from last '[' if present
    tag_start = line.rfind("[")
    if tag_start != -1:
        tag_raw = line[tag_start:].strip().strip("[]")
        rest = line[first_space:tag_start]
    else:
        tag_raw = ""
        rest = line[first_space:]

    wkt_str = rest.strip()
    if not wkt_str:
        raise ValueError("empty WKT field")

    tag_dict = {}
    for item in tag_raw.split(","):
        item = item.strip()
        if "#" in item:
            k, _, v = item.partition("#")
            tag_dict[k.strip()] = v.strip()

    return feat_id, wkt_str, tag_dict


def load_geometries(path: str) -> Tuple[List, List[str], List[dict]]:
    """
    Load geometries, IDs, and tags.

    Supported:
      - .tsv : <id> <WKT> [tags]
      - .wkt / .txt : one WKT per line
      - others via GeoPandas
    """
    geometries = []
    ids = []
    tags = []
    skipped = 0

    if path.endswith(".tsv"):
        with open(path, encoding="utf-8") as fh:
            for lineno, line in enumerate(fh, 1):
                try:
                    feat_id, wkt_str, tag_dict = _parse_tsv_line(line)
                except ValueError as exc:
                    if line.strip():
                        print(f"[skip] line {lineno}: {exc}")
                    skipped += 1
                    continue

                try:
                    geom = shapely_wkt.loads(wkt_str)
                except Exception as exc:
                    print(f"[skip] line {lineno} id={feat_id}: WKT error — {exc}")
                    print(f"       WKT preview: {wkt_str[:200]}")
                    skipped += 1
                    continue

                try:
                    geom = _geom_to_polygon(geom)
                except Exception as exc:
                    print(f"[skip] line {lineno} id={feat_id}: geometry conversion error — {exc}")
                    print(f"       WKT preview: {wkt_str[:200]}")
                    skipped += 1
                    continue

                if not geom.is_valid:
                    geom = geom.buffer(0)

                if geom.is_empty:
                    skipped += 1
                    continue

                geometries.append(geom)
                ids.append(feat_id)
                tags.append(tag_dict)

    elif path.endswith(".wkt") or path.endswith(".txt"):
        with open(path, encoding="utf-8") as fh:
            for lineno, line in enumerate(fh, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    geom = shapely_wkt.loads(line)
                    geom = _geom_to_polygon(geom)

                    if not geom.is_valid:
                        geom = geom.buffer(0)

                    if geom.is_empty:
                        raise ValueError("empty geometry after fix")

                    geometries.append(geom)
                    ids.append(str(lineno))
                    tags.append({})
                except Exception as exc:
                    print(f"[skip] line {lineno}: {exc}")
                    skipped += 1

    else:
        import geopandas as gpd

        gdf = gpd.read_file(path)
        for i, row in gdf.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty:
                skipped += 1
                continue

            try:
                geom = _geom_to_polygon(geom)

                if not geom.is_valid:
                    geom = geom.buffer(0)

                if geom.is_empty:
                    raise ValueError("empty geometry after fix")

                geometries.append(geom)
                ids.append(str(i))
                tags.append({})
            except Exception as exc:
                print(f"[skip] row {i}: {exc}")
                skipped += 1

    print(f"Loaded {len(geometries):,} valid geometries ({skipped} skipped) from {path}")
    return geometries, ids, tags


def to_multipolygon(geom) -> MultiPolygon:
    if geom.geom_type == "Polygon":
        return MultiPolygon([geom])
    elif geom.geom_type == "MultiPolygon":
        return geom
    raise ValueError(f"Unsupported geometry type: {geom.geom_type}")


print("Cell 1 loaded successfully.")

Cell 1 loaded successfully.


In [3]:
# ============================================================
# CELL 1B: Silent loader + summary only
# ============================================================

from collections import Counter

DATA_PATH = "/raid/ruban/data/parks.tsv"

def load_geometries_silent(path: str):
    geometries = []
    ids = []
    tags = []

    skipped = 0
    skip_reasons = Counter()

    with open(path, encoding="utf-8") as fh:
        for lineno, line in enumerate(fh, 1):
            try:
                feat_id, wkt_str, tag_dict = _parse_tsv_line(line)
            except Exception:
                skipped += 1
                skip_reasons["parse_error"] += 1
                continue

            try:
                geom = shapely_wkt.loads(wkt_str)
            except Exception:
                skipped += 1
                skip_reasons["wkt_parse_error"] += 1
                continue

            try:
                geom = _geom_to_polygon(geom)
            except Exception as exc:
                skipped += 1
                msg = str(exc).lower()

                if "too short" in msg:
                    skip_reasons["degenerate_linestring"] += 1
                elif "empty" in msg:
                    skip_reasons["empty_geometry"] += 1
                elif "closed ring too short" in msg:
                    skip_reasons["short_closed_ring"] += 1
                else:
                    skip_reasons["geometry_conversion_error"] += 1
                continue

            try:
                if not geom.is_valid:
                    geom = geom.buffer(0)
                if geom.is_empty:
                    skipped += 1
                    skip_reasons["empty_after_fix"] += 1
                    continue
            except Exception:
                skipped += 1
                skip_reasons["post_fix_error"] += 1
                continue

            geometries.append(geom)
            ids.append(feat_id)
            tags.append(tag_dict)

    print("=== LOAD SUMMARY ===")
    print(f"Valid geometries : {len(geometries):,}")
    print(f"Skipped          : {skipped:,}")
    print(f"Kept ratio       : {len(geometries) / (len(geometries) + skipped):.4f}")

    print("\n=== SKIP REASONS ===")
    for k, v in skip_reasons.most_common():
        print(f"{k:24s}: {v:,}")

    return geometries, ids, tags, skip_reasons


# Run this
geometries, ids, tags, skip_reasons = load_geometries_silent(DATA_PATH)

print("\n=== GEOMETRY TYPE SUMMARY ===")
type_counts = Counter(g.geom_type for g in geometries)
for k, v in sorted(type_counts.items()):
    print(f"{k:15s}: {v:,}")

=== LOAD SUMMARY ===
Valid geometries : 234,195
Skipped          : 252
Kept ratio       : 0.9989

=== SKIP REASONS ===
degenerate_linestring   : 196
empty_geometry          : 46
wkt_parse_error         : 10

=== GEOMETRY TYPE SUMMARY ===
MultiPolygon   : 114
Polygon        : 234,081


In [5]:
# ============================================================
# CELL 2: Extract exterior rings (PolyMP input)
# ============================================================

import numpy as np

rings = []
valid_ids = []

for g, pid in zip(geometries, ids):
    try:
        if g.geom_type == "MultiPolygon":
            g = max(g.geoms, key=lambda x: x.area)

        if g.geom_type != "Polygon":
            continue

        coords = np.array(g.exterior.coords)

        if len(coords) < 4:
            continue

        rings.append(coords)
        valid_ids.append(pid)

    except:
        continue

print("=== RING SUMMARY ===")
print("Total rings:", len(rings))
print("Sample shape:", rings[0].shape)

=== RING SUMMARY ===
Total rings: 234195
Sample shape: (57, 2)


In [41]:
# ============================================================
# CELL 3: Build geometry-aware features (CRITICAL)
# ============================================================

import numpy as np

def normalize_polygon(x):
    # center
    x = x - x.mean(axis=0)

    # scale (max distance)
    scale = np.linalg.norm(x, axis=1).max()
    if scale > 0:
        x = x / scale

    return x


def build_features(coords):
    # -------------------------
    # ORIGINAL coords (for spatial features) ✅
    # -------------------------
    orig = coords.copy()

    # -------------------------
    # NORMALIZED coords (for shape learning)
    # -------------------------
    x = normalize_polygon(coords)

    N = len(x) - 1

    feats = []
    edges = []

    # -------------------------
    # GLOBAL features (FROM ORIGINAL coords) 🔥
    # -------------------------
    cx = orig[:, 0].mean()
    cy = orig[:, 1].mean()

    minx, miny = orig.min(axis=0)
    maxx, maxy = orig.max(axis=0)

    w = maxx - minx
    h = maxy - miny

    # -------------------------
    # BUILD NODE FEATURES
    # -------------------------
    for i in range(N):
        p = x[i]
        q = x[(i + 1) % N]

        dx, dy = q - p
        edge_len = np.sqrt(dx * dx + dy * dy)
        angle = np.arctan2(dy, dx)

        feats.append([
            p[0], p[1],     # normalized position
            edge_len,
            angle,
            dx, dy,

            # 🔥 ADD THESE (global spatial features)
            cx, cy, w, h
        ])

        edges.append([i, (i + 1) % N])
        edges.append([(i + 1) % N, i])

    return np.array(feats, dtype=np.float32), np.array(edges).T


features = []
edge_indices = []

for r in rings:
    f, e = build_features(r)
    features.append(f)
    edge_indices.append(e)

print("=== FEATURE SUMMARY ===")
print("Feature shape example:", features[0].shape)
print("Edge shape example   :", edge_indices[0].shape)

=== FEATURE SUMMARY ===
Feature shape example: (56, 10)
Edge shape example   : (2, 112)


In [42]:
# ============================================================
# CELL 10: Load GT (Jaccard)
# ============================================================

import os

GT_PATH = "/raid/ruban/groundtruth/pk-query-187019"

gt = {}

for fname in os.listdir(GT_PATH):
    path = os.path.join(GT_PATH, fname)

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split(",")

            if len(parts) < 2:
                continue

            q = int(parts[0])
            neighbors = list(map(int, parts[1:]))

            gt[q] = neighbors

print("GT queries:", len(gt))

GT queries: 44666


In [43]:
# ============================================================
# CELL 4: Subset + PyG graph creation (FAST + SAFE)
# ============================================================

import torch
from torch_geometric.data import Data
import random

# ---- SUBSET (critical for speed)
MAX_SAMPLES = 15000   # adjust if needed

# ============================================================
# GT-aligned subset (FIX)
# ============================================================

gt_ids = set(gt.keys())

# step 1: get GT indices
indices = [i for i, pid in enumerate(valid_ids) if pid in gt_ids]

# step 2: fill remaining randomly
extra_needed = MAX_SAMPLES - len(indices)

remaining = [i for i in range(len(valid_ids)) if i not in indices]
random.shuffle(remaining)

indices += remaining[:extra_needed]

print("GT-aligned subset:", len(indices))

graphs = []

for idx in indices:
    g = Data(
        x=torch.tensor(features[idx]),
        edge_index=torch.tensor(edge_indices[idx], dtype=torch.long),
        poly_id=int(idx)
    )
    graphs.append(g)

print("=== GRAPH SUMMARY ===")
print("Total graphs:", len(graphs))
print("Sample graph:", graphs[0])

GT-aligned subset: 15000
=== GRAPH SUMMARY ===
Total graphs: 15000
Sample graph: Data(x=[19, 10], edge_index=[2, 38], poly_id=105690)


In [50]:
# ============================================================
# CELL 5: PolyMP Model
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_mean
from torch_geometric.nn import global_mean_pool

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class PolyMPConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()

        self.mlp_msg = nn.Sequential(
            nn.Linear(in_dim * 2, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

        self.mlp_update = nn.Sequential(
            nn.Linear(in_dim + out_dim, out_dim),
            nn.ReLU()
        )

    def forward(self, x, edge_index):
        row, col = edge_index

        m = torch.cat([x[row], x[col]], dim=1)
        m = self.mlp_msg(m)

        agg = scatter_mean(m, col, dim=0, dim_size=x.size(0))

        out = self.mlp_update(torch.cat([x, agg], dim=1))
        return out


class PolygonEncoderPolyMP(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, out_dim=128):
        super().__init__()

        self.conv1 = PolyMPConv(in_dim, hidden_dim)
        self.conv2 = PolyMPConv(hidden_dim, hidden_dim)
        self.conv3 = PolyMPConv(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(0.1)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.conv2(x, edge_index)
        x = self.conv3(x, edge_index)

        x = self.dropout(x)

        x = global_mean_pool(x, batch)

        x = self.proj(x)

        x = F.normalize(x, dim=1)

        return x


model = PolygonEncoderPolyMP(in_dim=10).to(DEVICE)

print(model)

PolygonEncoderPolyMP(
  (conv1): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=20, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=74, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (conv2): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (conv3): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (dropout): Dropout

In [51]:
# ============================================================
# CELL 6: Triplet dataset (self-supervised)
# ============================================================

import random
import math
import torch
from torch_geometric.data import Data


def rotate(x, angle):
    c, s = math.cos(angle), math.sin(angle)
    R = torch.tensor([[c, -s], [s, c]], dtype=x.dtype)
    return x @ R.T


def augment_graph(g):
    x = g.x.clone()

    # only rotate XY (first 2 dims)
    coords = x[:, :2]

    angle = random.uniform(-0.3, 0.3)
    coords = rotate(coords, angle)

    scale = random.uniform(0.9, 1.1)
    coords = coords * scale

    x[:, :2] = coords

    return Data(x=x, edge_index=g.edge_index, poly_id=g.poly_id)


class TripletDataset:
    def __init__(self, graphs):
        self.graphs = graphs

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        anchor = self.graphs[idx]

        # positive = augmented version
        positive = augment_graph(anchor)

        # negative = random different polygon
        neg_idx = random.randint(0, len(self.graphs) - 1)
        while neg_idx == idx:
            neg_idx = random.randint(0, len(self.graphs) - 1)

        negative = self.graphs[neg_idx]

        return anchor, positive, negative


# split
split = int(0.9 * len(graphs))
train_graphs = graphs[:split]
val_graphs = graphs[split:]

train_triplet_ds = TripletDataset(train_graphs)
val_triplet_ds = TripletDataset(val_graphs)

print("Triplet dataset ready")

Triplet dataset ready


In [52]:
# ============================================================
# CELL 7: Triplet Training (CORE LEARNING)
# ============================================================

import torch
import numpy as np
from torch_geometric.loader import DataLoader

# ---- loaders (simple, fast)
def make_triplet_batches(dataset, batch_size=32):
    for start in range(0, len(dataset), batch_size):
        batch = [dataset[i] for i in range(start, min(start + batch_size, len(dataset)))]

        a_list = [x[0] for x in batch]
        p_list = [x[1] for x in batch]
        n_list = [x[2] for x in batch]

        batch_a = next(iter(DataLoader(a_list, batch_size=len(a_list))))
        batch_p = next(iter(DataLoader(p_list, batch_size=len(p_list))))
        batch_n = next(iter(DataLoader(n_list, batch_size=len(n_list))))

        yield batch_a, batch_p, batch_n


# ---- optimizer + loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

MARGIN = 0.2
EPOCHS = 10
BATCH_SIZE = 32


for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for batch_a, batch_p, batch_n in make_triplet_batches(train_triplet_ds, BATCH_SIZE):
        batch_a = batch_a.to(DEVICE)
        batch_p = batch_p.to(DEVICE)
        batch_n = batch_n.to(DEVICE)

        h_a = model(batch_a)
        h_p = model(batch_p)
        h_n = model(batch_n)

        # cosine similarity (since normalized)
        pos_sim = (h_a * h_p).sum(dim=1)
        neg_sim = (h_a * h_n).sum(dim=1)

        loss = torch.relu(MARGIN - pos_sim + neg_sim).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())

    # ---- validation
    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch_a, batch_p, batch_n in make_triplet_batches(val_triplet_ds, BATCH_SIZE):
            batch_a = batch_a.to(DEVICE)
            batch_p = batch_p.to(DEVICE)
            batch_n = batch_n.to(DEVICE)

            h_a = model(batch_a)
            h_p = model(batch_p)
            h_n = model(batch_n)

            pos_sim = (h_a * h_p).sum(dim=1)
            neg_sim = (h_a * h_n).sum(dim=1)

            loss = torch.relu(MARGIN - pos_sim + neg_sim).mean()
            val_losses.append(loss.item())

    print(f"Epoch {epoch:02d} | train_loss={np.mean(train_losses):.4f} | val_loss={np.mean(val_losses):.4f}")

Epoch 01 | train_loss=0.0148 | val_loss=0.0076
Epoch 02 | train_loss=0.0088 | val_loss=0.0081
Epoch 03 | train_loss=0.0078 | val_loss=0.0069
Epoch 04 | train_loss=0.0077 | val_loss=0.0054
Epoch 05 | train_loss=0.0070 | val_loss=0.0057
Epoch 06 | train_loss=0.0063 | val_loss=0.0040
Epoch 07 | train_loss=0.0059 | val_loss=0.0061
Epoch 08 | train_loss=0.0058 | val_loss=0.0047
Epoch 09 | train_loss=0.0060 | val_loss=0.0056
Epoch 10 | train_loss=0.0060 | val_loss=0.0049


In [53]:
# ============================================================
# CELL 8: Extract embeddings
# ============================================================

import torch
from torch_geometric.loader import DataLoader

model.eval()

loader = DataLoader(graphs, batch_size=128, shuffle=False)

all_emb = []
all_ids = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(DEVICE)

        h = model(batch)   # [B,128]

        all_emb.append(h.cpu())
        all_ids.extend(batch.poly_id.tolist())

H = torch.cat(all_emb, dim=0)

print("Embedding shape:", H.shape)

Embedding shape: torch.Size([15000, 128])


In [60]:
from shapely.geometry import Polygon

def compute_iou(idx1, idx2):
    p1 = Polygon(rings[idx1])
    p2 = Polygon(rings[idx2])

    if not p1.is_valid or not p2.is_valid:
        return 0.0

    inter = p1.intersection(p2).area
    union = p1.union(p2).area

    if union == 0:
        return 0.0

    return inter / union

In [61]:
H_np = H.detach().cpu().numpy()

def get_topk_rerank(query_idx, k=10, cand_k=200):
    q = H_np[query_idx]

    sims = H_np @ q
    sims[query_idx] = -1e9

    # Step 1: retrieve candidates
    cand_idx = np.argsort(-sims)[:cand_k]

    # Step 2: re-rank using IoU
    scores = []
    for c in cand_idx:
        iou = compute_iou(query_idx, c)
        scores.append((c, iou))

    scores.sort(key=lambda x: -x[1])

    return [c for c, _ in scores[:k]]

In [62]:
# ============================================================
# CELL 11: ID → index mapping
# ============================================================

id_to_idx = {pid: i for i, pid in enumerate(all_ids)}

valid_queries = [q for q in gt if q in id_to_idx]

print("Valid GT queries:", len(valid_queries))

Valid GT queries: 2815


In [63]:
print("Sample GT IDs:", list(gt.keys())[:5])
print("Sample dataset IDs:", all_ids[:5])

Sample GT IDs: [201041, 201042, 201043, 201044, 201045]
Sample dataset IDs: [105690, 208572, 10369, 195531, 95577]


In [64]:
gt_sample = list(gt.keys())[:50]

found = 0
for q in gt_sample:
    if q in all_ids:
        found += 1

print("Matches in first 50 GT:", found)

Matches in first 50 GT: 4


In [75]:
# ============================================================
# CORRECT LOADER (INDEX-AWARE)
# ============================================================

import os
import glob
import re
import numpy as np
from tqdm import tqdm

ENCODING_DIR = "/raid/ruban/encodings/pk-real0.002"

pattern = os.path.join(ENCODING_DIR, "real_*.txt")
files = glob.glob(pattern)

# Extract numeric start ID and sort properly
def get_start_id(f):
    return int(re.search(r"real_(\d+)\.txt", f).group(1))

files = sorted(files, key=get_start_id)

# Allocate full array
POLY_COUNT = 234195   # use your known value
encodings = [None] * POLY_COUNT

def parse_line(line):
    toks = line.strip().split()
    ids, vals = [], []

    for tok in toks:
        if ":" in tok:
            k, v = tok.split(":")
            v = float(v)
            if v != 0:
                ids.append(int(k))
                vals.append(v)

    return np.array(ids), np.array(vals)


print("Loading encodings correctly...")

for fpath in tqdm(files):
    start_id = get_start_id(fpath)

    with open(fpath, "r") as f:
        for i, line in enumerate(f):
            poly_id = start_id + i
            encodings[poly_id] = parse_line(line)

print("Done.")
print("Total:", len(encodings))

Loading encodings correctly...


100%|██████████| 400/400 [03:32<00:00,  1.89it/s]

Done.
Total: 234195


In [89]:
# ============================================================
# CELL X: Define DB / Query split (CRITICAL)
# ============================================================

DATA_END = int(0.8 * len(encodings))

db_indices = list(range(0, DATA_END))
query_indices = list(range(DATA_END, len(encodings)))

print("DB size:", len(db_indices))
print("Query size:", len(query_indices))

DB size: 187356
Query size: 46839


In [90]:
# ============================================================
# CLEAN: remove None entries
# ============================================================

valid_mask = [e is not None for e in encodings]

valid_indices_full = [i for i, v in enumerate(valid_mask) if v]
encodings_clean = [encodings[i] for i in valid_indices_full]

print("Valid encodings:", len(encodings_clean))
print("Removed None:", len(encodings) - len(encodings_clean))

Valid encodings: 233773
Removed None: 422


In [91]:
import glob
import os

pattern = os.path.join(ENCODING_DIR, "real_*.txt")
files = glob.glob(pattern)

print("Files found:", len(files))
print("Sample files:", files[:5])

Files found: 400
Sample files: ['/raid/ruban/encodings/pk-real0.002/real_24360.txt', '/raid/ruban/encodings/pk-real0.002/real_172864.txt', '/raid/ruban/encodings/pk-real0.002/real_156344.txt', '/raid/ruban/encodings/pk-real0.002/real_212576.txt', '/raid/ruban/encodings/pk-real0.002/real_127144.txt']


In [92]:
# ============================================================
# ALIGN encodings with subset
# ============================================================

encodings_subset = []
indices_clean = []

for i in indices:
    if encodings[i] is not None:
        encodings_subset.append(encodings[i])
        indices_clean.append(i)

indices = indices_clean  # IMPORTANT

In [97]:
# ============================================================
# CELL 9.5: Weighted Jaccard + retrieval
# ============================================================

def weighted_jaccard(a, b):
    ids_a, vals_a = a
    ids_b, vals_b = b

    dict_a = {k: v for k, v in zip(ids_a, vals_a)}
    dict_b = {k: v for k, v in zip(ids_b, vals_b)}

    keys = set(dict_a.keys()) | set(dict_b.keys())

    num = 0.0
    den = 0.0

    for k in keys:
        va = dict_a.get(k, 0.0)
        vb = dict_b.get(k, 0.0)

        num += min(va, vb)
        den += max(va, vb)

    return num / den if den > 0 else 0.0


def get_topk_jaccard(query_id, k=10):
    q = encodings[query_id]

    scores = []
    for i in db_indices:   # ✅ ONLY DATABASE
        if encodings[i] is None:
            continue   # 🔥 skip bad rows
        
        s = weighted_jaccard(q, encodings[i])
        scores.append((i, s))

    scores.sort(key=lambda x: -x[1])
    return [i for i, _ in scores[:k]]

In [98]:
# ============================================================
# Mapping: global → local (subset)
# ============================================================

global_to_local = {g: i for i, g in enumerate(indices)}

print("Mapping size:", len(global_to_local))

Mapping size: 14972


In [99]:
# ============================================================
# FINAL: Recall@10 Evaluation (REAL GT)
# ============================================================

import numpy as np

H_np = H.detach().cpu().numpy()

def get_topk_jaccard(query_global_id, k=10):
    q = encodings[query_global_id]

    scores = []
    for i in range(len(encodings)):
        if i == query_global_id:
            continue

        s = weighted_jaccard(q, encodings[i])
        scores.append((i, s))

    scores.sort(key=lambda x: -x[1])

    return [i for i, _ in scores[:k]]


def recall_at_k(k=10, max_queries=200):
    scores = []

    queries = list(gt.keys())[:max_queries]

    for q in queries:
        pred = get_topk_jaccard(q, k)

        pred_set = set(pred)
        gt_set = set(gt[q][:k])

        overlap = len(pred_set & gt_set)
        scores.append(overlap / k)

    return np.mean(scores)


r10 = recall_at_k(10)

print("\n=== FINAL RESULT ===")
print(f"Recall@10: {r10:.4f}")

TypeError: cannot unpack non-iterable NoneType object

In [66]:
from shapely.geometry import Polygon

q = valid_queries[0]
q_idx = id_to_idx[q]

print("Query:", q)

# take 5 random candidates
for i in range(5):
    c_idx = np.random.randint(0, len(all_ids))

    p1 = Polygon(rings[q])
    p2 = Polygon(rings[all_ids[c_idx]])

    inter = p1.intersection(p2).area
    union = p1.union(p2).area

    iou = inter / union if union > 0 else 0

    print(f"Candidate {c_idx} → IoU: {iou}")

Query: 201042
Candidate 11784 → IoU: 0.0
Candidate 9133 → IoU: 0.0
Candidate 5637 → IoU: 0.0
Candidate 14294 → IoU: 0.0
Candidate 5597 → IoU: 0.0


In [96]:
sum(1 for e in encodings if e is None)

422

In [88]:
q = valid_queries[0]

# take first GT neighbor
gt_n = gt[q][0]

# compute similarity
sim = weighted_jaccard(encodings[q], encodings[gt_n])

print("GT similarity:", sim)

GT similarity: 0.0


In [100]:
# ============================================================
# FINAL CLEAN EVAL: Exact weighted Jaccard against GT
# ============================================================

import numpy as np

# GT is only for query split; DB is first 80%
DATA_END = int(0.8 * len(encodings))
db_indices = list(range(0, DATA_END))

def weighted_jaccard(a, b):
    if a is None or b is None:
        return 0.0

    ids_a, vals_a = a
    ids_b, vals_b = b

    dict_a = {k: v for k, v in zip(ids_a, vals_a)}
    dict_b = {k: v for k, v in zip(ids_b, vals_b)}

    keys = set(dict_a.keys()) | set(dict_b.keys())

    num = 0.0
    den = 0.0

    for k in keys:
        va = dict_a.get(k, 0.0)
        vb = dict_b.get(k, 0.0)
        num += min(va, vb)
        den += max(va, vb)

    return num / den if den > 0 else 0.0


def get_topk_jaccard(query_id, k=10):
    q = encodings[query_id]
    if q is None:
        return []

    scores = []
    for i in db_indices:
        if encodings[i] is None:
            continue
        s = weighted_jaccard(q, encodings[i])
        scores.append((i, s))

    scores.sort(key=lambda x: -x[1])
    return [i for i, _ in scores[:k]]


def recall_at_k(k=10, max_queries=200):
    scores = []

    # use GT queries directly
    queries = list(gt.keys())[:max_queries]

    for q in queries:
        pred = get_topk_jaccard(q, k)
        if not pred:
            continue

        pred_set = set(pred)
        gt_set = set(gt[q][:k])

        overlap = len(pred_set & gt_set)
        scores.append(overlap / k)

    return np.mean(scores) if scores else 0.0


# quick sanity check: GT neighbor should not be zero-sim if encoding folder matches GT
q0 = list(gt.keys())[0]
gt0 = gt[q0][0]
print("Sanity GT similarity:", weighted_jaccard(encodings[q0], encodings[gt0]))

r10 = recall_at_k(10)

print("\n=== FINAL RESULT ===")
print(f"Recall@10: {r10:.4f}")

Sanity GT similarity: 0.0


KeyboardInterrupt: 

In [101]:
# ============================================================
# DEBUG: inspect one GT query against its first GT neighbor
# ============================================================

q = list(gt.keys())[0]
n = gt[q][0]

print("q:", q, "n:", n)

print("encodings[q] is None:", encodings[q] is None)
print("encodings[n] is None:", encodings[n] is None)

ids_q, vals_q = encodings[q]
ids_n, vals_n = encodings[n]

print("len(ids_q):", len(ids_q), "len(ids_n):", len(ids_n))
print("first 10 ids_q:", ids_q[:10])
print("first 10 ids_n:", ids_n[:10])

overlap = len(set(ids_q.tolist()) & set(ids_n.tolist()))
print("raw id overlap:", overlap)

print("weighted_jaccard:", weighted_jaccard(encodings[q], encodings[n]))

q: 201041 n: 116803
encodings[q] is None: False
encodings[n] is None: False
len(ids_q): 0 len(ids_n): 0
first 10 ids_q: []
first 10 ids_n: []
raw id overlap: 0
weighted_jaccard: 0.0


In [ ]:
# ============================================================
# 50K BINARY ENCODINGS + GT EVAL (FAST + CORRECT)
# ============================================================

import os
import glob
import re
import numpy as np
from tqdm import tqdm
import heapq

ENCODING_DIR = "/raid/ruban/encodings/pk-50k0.002"
GT_DIR = "/raid/ruban/groundtruth/pk-query-50k"

POLY_COUNT = 50000
DATA_END = 40000
QUERY_START = 40000
QUERY_END = 50000

# ----------------------------
# load GT
# ----------------------------
gt = {}
for fname in sorted(os.listdir(GT_DIR)):
    fpath = os.path.join(GT_DIR, fname)
    with open(fpath, "r") as f:
        for line in f:
            parts = [x.strip() for x in line.strip().split(",") if x.strip()]
            if len(parts) < 2:
                continue
            q = int(parts[0])
            nbrs = list(map(int, parts[1:]))
            gt[q] = nbrs

print("GT queries loaded:", len(gt))

# ----------------------------
# binary encoding parser
# supports:
#   "1 5 10 12"
#   "1:1 5:1 10:1"
#   "1:0.3 5:1.0 ..."  -> keep only active ids
# ----------------------------
def parse_binary_line(line, zero_tol=1e-12):
    toks = line.strip().split()

    if not toks:
        return np.empty(0, dtype=np.int32)

    # idx:value format
    if ":" in toks[0]:
        ids = []
        for tok in toks:
            k, v = tok.split(":", 1)
            if abs(float(v)) > zero_tol:
                ids.append(int(k))
        return np.asarray(ids, dtype=np.int32)

    # plain active IDs
    return np.asarray([int(x) for x in toks], dtype=np.int32)

def get_start_id(path):
    base = os.path.basename(path)
    m = re.search(r"(\d+)", base)
    if m is None:
        raise ValueError(f"Could not parse start id from {base}")
    return int(m.group(1))

# ----------------------------
# load encodings into full 50k array
# ----------------------------
pattern = os.path.join(ENCODING_DIR, "*.txt")
files = sorted(glob.glob(pattern), key=get_start_id)

encodings = [None] * POLY_COUNT

for fpath in tqdm(files, desc="Loading 50k encodings"):
    start_id = get_start_id(fpath)
    with open(fpath, "r") as f:
        for row_idx, line in enumerate(f):
            pid = start_id + row_idx
            if 0 <= pid < POLY_COUNT:
                encodings[pid] = parse_binary_line(line)

none_count = sum(1 for e in encodings if e is None)
print("Total encodings:", len(encodings))
print("Missing rows:", none_count)

# ----------------------------
# precompute sets once (major speedup)
# ----------------------------
encodings_sets = []
for e in encodings:
    if e is None:
        encodings_sets.append(None)
    else:
        encodings_sets.append(set(e.tolist()))

# ----------------------------
# plain Jaccard on active-cell sets
# ----------------------------
def jaccard_sets(a, b):
    if a is None or b is None:
        return 0.0
    if not a and not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union > 0 else 0.0

# sanity check
q0 = sorted(gt.keys())[0]
n0 = gt[q0][0]
print("Sanity:")
print("q0:", q0, "len:", 0 if encodings[q0] is None else len(encodings[q0]))
print("n0:", n0, "len:", 0 if encodings[n0] is None else len(encodings[n0]))
print("J(q0, n0):", jaccard_sets(encodings_sets[q0], encodings_sets[n0]))

# ----------------------------
# retrieve ONLY from DB split
# query IDs are in [40000, 49999]
# db IDs are in [0, 39999]
# use heap for speed
# ----------------------------
db_indices = range(0, DATA_END)

def get_topk_jaccard(query_id, k=10):
    qset = encodings_sets[query_id]
    if qset is None:
        return []

    scores = []
    for i in db_indices:
        eset = encodings_sets[i]
        if eset is None:
            continue
        s = jaccard_sets(qset, eset)
        scores.append((s, i))

    topk = heapq.nlargest(k, scores)
    return [i for s, i in topk]

# ----------------------------
# recall@10
# start with max_queries=200 for speed
# then scale to 1000 or full once validated
# ----------------------------
def recall_at_k(k=10, max_queries=200):
    queries = sorted(gt.keys())
    if max_queries is not None:
        queries = queries[:max_queries]

    vals = []
    for q in tqdm(queries, desc=f"Recall@{k} eval"):
        pred = get_topk_jaccard(q, k)
        if not pred:
            continue

        pred_set = set(pred)
        gt_set = set(gt[q][:k])
        vals.append(len(pred_set & gt_set) / k)

    return float(np.mean(vals)) if vals else 0.0



GT queries loaded: 9417


Loading 50k encodings: 100%|██████████| 200/200 [00:35<00:00,  5.71it/s]


Total encodings: 50000
Missing rows: 0
Sanity:
q0: 40000 len: 8713
n0: 26769 len: 8700
J(q0, n0): 0.970465089962657


Recall@10 eval:   8%|▊         | 17/200 [02:10<23:22,  7.67s/it]


KeyboardInterrupt: 

In [104]:
r10_200 = recall_at_k(10, max_queries=10)
print("\n=== FINAL RESULT ===")
print(f"Recall@10 (200 queries): {r10_200:.4f}")

# optional scale-up after sanity:
# r10_1000 = recall_at_k(10, max_queries=1000)
# print(f"Recall@10 (1000 queries): {r10_1000:.4f}")

Recall@10 eval: 100%|██████████| 10/10 [01:06<00:00,  6.66s/it]


=== FINAL RESULT ===
Recall@10 (200 queries): 0.6600
